# Notebook 27 — Sensitivity Analysis: Oster Bounds and Robustness Values
### Heterogeneous Treatment Effects in Mortgage Lending

**Author:** Rajveer Singh Pall
**Institution:** Gyan Ganga Institute of Technology and Sciences

---

## Purpose

Provides formal omitted variable bias analysis for the AUS channel claim.
The paper reports that manual underwriting is associated with an 8.6 pp
larger racial penalty than automated underwriting. This notebook formalises
the robustness of that claim using:

1. **Oster (2019) delta bounds**: How strong must selection on unobservables be,
   relative to selection on observables, to fully explain the 8.6 pp gap?
   delta > 1 is required for the association to be entirely spurious.

2. **Cinelli & Hazlett (2020) robustness values**: What minimum partial R-squared
   of an omitted confounder would be needed to reduce the AUS effect to zero?

3. **Partial R-squared benchmarking**: Compare against observed covariates.

**INPUT:** Model-fit statistics from NB19 (must be extracted manually first)
**OUTPUTS:** `nb27_sensitivity_analysis.csv`, `nb27_sensitivity_analysis.png`
**RUNTIME:** ~5-10 minutes (analytical)

---

## IMPORTANT: Before running this notebook

Run NB19 with an additional OLS regression step (linear regression of
approved ~ black*manual_aus + X_FULL) and record:
- beta_uncontrolled: coefficient from bivariate OLS (approved ~ black + manual_aus)
- r2_uncontrolled: R-squared from that bivariate model
- r2_controlled: R-squared from the full 30-covariate model
- t_stat_aus: t-statistic of the manual_aus coefficient in the full model

Then replace the PLACEHOLDER values in Cells 1 and 2 below.


In [ ]:
# CELL 1 - OSTER (2019) DELTA COMPUTATION
# ============================================================================
# Oster (2019): delta = (beta_controlled * (R2_controlled - R2_uncontrolled)) /
#                       ((beta_uncontrolled - beta_controlled) * (R2_max - R2_controlled))
#
# delta > 1: unobservables must dominate observables to nullify the AUS effect
# ============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

BASE_DIR    = Path('D:/CATE-HMDA-Heterogeneous-Effects')
TABLES_DIR  = BASE_DIR / 'outputs' / 'tables'
FIGURES_DIR = BASE_DIR / 'outputs' / 'figures'

# =========================================================
# TODO: REPLACE WITH ACTUAL VALUE FROM NB19 OLS OUTPUT
# beta_uncontrolled: run OLS of (approved ~ black + manual_aus) — no other controls
beta_uncontrolled = -0.150   # PLACEHOLDER
# =========================================================

# =========================================================
# TODO: REPLACE WITH ACTUAL VALUE FROM NB19 OLS OUTPUT
# r2_uncontrolled: R-squared from bivariate OLS above
r2_uncontrolled   = 0.04     # PLACEHOLDER
# =========================================================

# =========================================================
# TODO: REPLACE WITH ACTUAL VALUE FROM NB19 OLS OUTPUT
# r2_controlled: R-squared from full model (approved ~ black + manual_aus + X_FULL)
r2_controlled     = 0.22     # PLACEHOLDER
# =========================================================

# This value is fixed from paper findings:
beta_controlled   = -0.086   # = 8.6 pp in decimal (from main results)

# R2_max: 1.3 * R2_controlled (Oster 2019 recommendation)
r2_max = min(1.3 * r2_controlled, 0.90)

print("="*70)
print("OSTER (2019) DELTA BOUNDS")
print("="*70)
print(f"Input parameters:")
print(f"  beta_uncontrolled : {beta_uncontrolled:.4f}  [PLACEHOLDER - update from NB19]")
print(f"  beta_controlled   : {beta_controlled:.4f}  [from main results]")
print(f"  R2_uncontrolled   : {r2_uncontrolled:.4f}  [PLACEHOLDER - update from NB19]")
print(f"  R2_controlled     : {r2_controlled:.4f}  [PLACEHOLDER - update from NB19]")
print(f"  R2_max            : {r2_max:.4f}  [1.3 x R2_controlled]")

delta_numerator   = beta_controlled * (r2_controlled - r2_uncontrolled)
delta_denominator = (beta_uncontrolled - beta_controlled) * (r2_max - r2_controlled)

delta = delta_numerator / delta_denominator if abs(delta_denominator) > 1e-10 else float('inf')

print(f"\nOster delta = {delta:.3f}")
if abs(delta) > 1.0:
    print(f"  |delta| = {abs(delta):.2f} > 1.0")
    print("  Unobservables would need to be MORE influential than observables")
    print("  to fully explain the 8.6 pp gap. Implausible with 30 controls.")
elif abs(delta) > 0.5:
    print(f"  |delta| = {abs(delta):.2f} (0.5 < delta < 1.0)")
    print("  Moderate confounding could explain part of the effect.")
else:
    print(f"  |delta| = {abs(delta):.2f} < 0.5")
    print("  Small confounding could explain the full gap. Exercise caution.")

print("\nSensitivity across R2_max assumptions:")
print(f"{'R2_max':>10} {'delta':>12} {'Interpretation':>25}")
print("-"*50)
for r2_max_test in [1.1 * r2_controlled, 1.3 * r2_controlled, 1.5 * r2_controlled, 0.8]:
    denom_test = (beta_uncontrolled - beta_controlled) * (r2_max_test - r2_controlled)
    delta_test = delta_numerator / denom_test if abs(denom_test) > 1e-10 else float('inf')
    interp = "Robust (|d|>1)" if abs(delta_test) > 1 else ("Moderate" if abs(delta_test) > 0.5 else "Fragile")
    print(f"{r2_max_test:>10.3f} {delta_test:>12.3f}   {interp}")


In [ ]:
# CELL 2 - CINELLI & HAZLETT (2020) ROBUSTNESS VALUES
# ============================================================================
# Robustness value (RV): minimum partial R-squared that an omitted confounder
# must have with BOTH AUS assignment and approval to reduce effect to zero.
# ============================================================================

# =========================================================
# TODO: REPLACE WITH ACTUAL VALUE FROM NB19 OLS OUTPUT
# t_stat_aus: t-statistic of manual_aus coefficient in full OLS model
t_stat_aus = 15.0     # PLACEHOLDER
# =========================================================

n_obs = 1_500_000
k_covariates = 31
df_model = n_obs - k_covariates - 1

partial_r2_aus = t_stat_aus**2 / (t_stat_aus**2 + df_model)
print(f"Partial R-squared of AUS in full model: {partial_r2_aus:.6f}")

rv_zero = partial_r2_aus / (1 + partial_r2_aus)
print(f"Robustness value (RV, effect->0)      : {rv_zero:.6f}")

# =========================================================
# TODO: REPLACE WITH ACTUAL VALUES FROM NB19 OLS OUTPUT
# These are the partial R-squared of each covariate in the full model
benchmarks = {
    'DTI ratio':        0.0180,   # PLACEHOLDER
    'LTV ratio':        0.0150,   # PLACEHOLDER
    'Log income':       0.0120,   # PLACEHOLDER
    'Loan purpose':     0.0080,   # PLACEHOLDER
    'Lender size':      0.0060,   # PLACEHOLDER
}
# =========================================================

print("\nBenchmark partial R-squared of observed covariates:")
print(f"{'Covariate':<25} {'Partial R2':>12} {'Exceeds RV?':>15}")
print("-"*54)
for cov, pr2 in sorted(benchmarks.items(), key=lambda x: -x[1]):
    exceeds = "YES" if pr2 > rv_zero else "No"
    print(f"  {cov:<23} {pr2:>12.4f} {exceeds:>15}")

print(f"\nRequired confounding partial R2 : {rv_zero:.6f}")
print(f"Max observed covariate partial R2: {max(benchmarks.values()):.4f}")

if max(benchmarks.values()) < rv_zero:
    print("\nROBUST: No observed covariate has partial R2 large enough to explain AUS effect.")
else:
    print("\nWARNING: Some observed covariates have partial R2 > RV.")

# Save results
sensitivity_df = pd.DataFrame([{
    'analysis': 'Oster delta',
    'value': delta,
    'threshold': 1.0,
    'robust': abs(delta) > 1.0,
    'note': 'PLACEHOLDER VALUES - update from NB19 OLS output before submission'
}, {
    'analysis': 'Cinelli-Hazlett RV (effect=0)',
    'value': rv_zero,
    'threshold': max(benchmarks.values()),
    'robust': max(benchmarks.values()) < rv_zero,
    'note': 'PLACEHOLDER VALUES - update from NB19 OLS output before submission'
}])
sensitivity_df.to_csv(TABLES_DIR / 'nb27_sensitivity_analysis.csv', index=False)
print("\nSaved: nb27_sensitivity_analysis.csv")
print("\n" + "="*70)
print("IMPORTANT: UPDATE ALL PLACEHOLDER VALUES BEFORE SUBMISSION")
print("="*70)
print("Values marked PLACEHOLDER must be updated with actual NB19 OLS output.")


In [ ]:
# CELL 3 - SENSITIVITY VISUALIZATION
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: Oster delta across R2_max assumptions
ax = axes[0]
r2_max_values = np.linspace(r2_controlled * 1.05, 0.9, 50)
delta_values  = []
for r2m in r2_max_values:
    denom = (beta_uncontrolled - beta_controlled) * (r2m - r2_controlled)
    d = delta_numerator / denom if abs(denom) > 1e-10 else float('inf')
    delta_values.append(min(max(d, -10), 15))

ax.plot(r2_max_values, delta_values, 'b-', linewidth=2.5)
ax.axhline(1.0, color='#E53935', linewidth=1.5, linestyle='--', label='delta=1 (robustness threshold)')
ax.axhline(-1.0, color='#E53935', linewidth=1.5, linestyle='--')
ax.set_xlabel('Assumed R2_max', fontsize=10)
ax.set_ylabel('Oster delta', fontsize=10)
ax.set_title('Oster Bounds: Selection Ratio Required\nto Nullify AUS Effect', fontsize=10)
ax.legend(fontsize=8)
ax.set_ylim(-5, 10)

# Right: Cinelli-Hazlett partial R2 space
ax2 = axes[1]
pr2_range = np.linspace(0, 0.05, 100)
PR2_T, PR2_Y = np.meshgrid(pr2_range, pr2_range)
reduction = np.sqrt(PR2_T * PR2_Y) / (rv_zero + 0.001)
reduction = np.clip(reduction, 0, 2)
cs = ax2.contourf(PR2_T, PR2_Y, reduction, levels=[0, 0.5, 1.0, 1.5, 2.0],
                  colors=['#C8E6C9', '#A5D6A7', '#EF9A9A', '#E53935', '#B71C1C'], alpha=0.7)
ax2.contour(PR2_T, PR2_Y, reduction, levels=[1.0], colors='black', linewidths=2)
for cov, pr2 in benchmarks.items():
    ax2.scatter(pr2, pr2, s=60, color='black', zorder=5)
    ax2.annotate(cov, (pr2, pr2), textcoords='offset points', xytext=(5, 5), fontsize=7)
ax2.set_xlabel("Confounder partial R2 with AUS type", fontsize=10)
ax2.set_ylabel("Confounder partial R2 with approval gap", fontsize=10)
ax2.set_title('Cinelli-Hazlett: Confounding Required\nto Nullify AUS Effect', fontsize=10)

plt.suptitle('Figure NB27: Formal Sensitivity Analysis -- AUS Channel\n(Note: Based on PLACEHOLDER values -- update before submission)', fontsize=10)
plt.tight_layout()
fig_path = FIGURES_DIR / 'nb27_sensitivity_analysis.png'
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
print(f"Saved: {fig_path.name}")
plt.show()
